In [3]:
# Loading dataset

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

def load_ascii_to_df(file_path):
    matrix = np.loadtxt(file_path)
    df_matrix = pd.DataFrame(matrix)
    df_long = df_matrix.stack().reset_index()
    df_long.columns = ['uid', 'iid', 'rating']
    df_long = df_long[df_long['rating'] > 0].reset_index(drop=True)
    df_long['uid'] = df_long['uid'].astype(int)
    df_long['iid'] = df_long['iid'].astype(int)
    
    return df_long

train_df = load_ascii_to_df('./data/train.ascii')
test_df = load_ascii_to_df('./data/test.ascii')

original_train_count = len(train_df)
merged = pd.merge(train_df, test_df[['uid', 'iid']], on=['uid', 'iid'], how='left', indicator=True)
train_df = merged[merged['_merge'] == 'left_only'].copy()

train_df.drop(columns=['_merge'], inplace=True)
train_df.reset_index(drop=True, inplace=True)

removed_count = original_train_count - len(train_df)

print(f"Removed {removed_count} duplicate entries from training set that were present in test set.")
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Removed 366 duplicate entries from training set that were present in test set.
Training set size: 6594
Test set size: 4640


In [4]:
# Feature processing

item_feature_map = {
    'gender': {0: 'men', 1: 'women'},
    'jackettype': {
        2: 'bomber', 3: 'cropped', 4: 'field', 5: 'fleece', 6: 'insulated',
        7: 'motorcycle', 8: 'other', 9: 'packable', 10: 'parkas', 11: 'pea',
        12: 'rain', 13: 'shells', 14: 'track', 15: 'trench', 16: 'vests', 17: 'waterproof'
    },
    'color': {
        18: 'beige', 19: 'black', 20: 'blue', 21: 'brown', 22: 'gray', 23: 'green',
        24: 'multi', 25: 'navy', 26: 'olive', 27: 'other', 28: 'pink', 29: 'purple', 30: 'red'
    },
    'onfrontpage': {31: 'yes', 32: 'no'}
}
user_feature_map = {
    'gender': {0: 'men', 1: 'women'},
    'age': {
        2: '20-30', 3: '30-40', 4: '40-50', 
        5: '50-60', 6: 'over 60', 7: 'under 20'
    },
    'location': {
        8: 'rural', 9: 'suburban', 10: 'urban'
    },
    'fashioninterest': {
        11: 'moderately', 12: 'not at all', 13: 'very'
    }
}

def process_features(file_path, feature_map):
    matrix = np.loadtxt(file_path)
    id_list = []
    for id, row in enumerate(matrix):
        data = {'vid': int(id)}
        for attr, idx_to_label in feature_map.items():
            relevant_indices = list(idx_to_label.keys())
            active_indices = [idx for idx in relevant_indices if row[idx] == 1]
            labels = [idx_to_label[idx] for idx in active_indices]
            data[attr] = ",".join(labels) if labels else None
        id_list.append(data)
    return pd.DataFrame(id_list)

item_features_df = process_features('./data/user_item_features/item_features.ascii', item_feature_map)
user_features_df = process_features('./data/user_item_features/user_features.ascii', user_feature_map)

# item_features_df = item_features_df.drop(columns=['onfrontpage'])

def concat_features(row):
    items = [f"{col}: {val}" for col, val in row.items() if col != "vid"]
    return ", ".join(items)

item_features_df['text_input'] = item_features_df.apply(concat_features, axis=1)
user_features_df['text_input'] = user_features_df.apply(concat_features, axis=1)
item_caption = item_features_df[['vid', 'text_input']]
user_caption = user_features_df[['vid', 'text_input']]

item_caption.to_csv('./rec_list/item_caption.csv', index=False)
user_caption.to_csv('./rec_list/user_caption.csv', index=False)

print("Item caption example:")
print(item_caption.head())
print("\nUser caption example:")
print(user_caption.head())

Item caption example:
   vid                                         text_input
0    0  gender: men, jackettype: bomber, color: other,...
1    1  gender: men, jackettype: insulated, color: blu...
2    2  gender: men, jackettype: other, color: olive, ...
3    3  gender: men, jackettype: rain, color: gray, on...
4    4  gender: men, jackettype: other, color: black, ...

User caption example:
   vid                                         text_input
0    0  gender: women, age: 20-30, location: urban, fa...
1    1  gender: women, age: 30-40, location: suburban,...
2    2  gender: women, age: 20-30, location: suburban,...
3    3  gender: women, age: 40-50, location: urban, fa...
4    4  gender: women, age: 30-40, location: suburban,...


In [3]:
# Save train uid-vid list

def build_uid_vid_list(data):
    grouped = data.groupby('uid')['iid'].apply(lambda seq: list(dict.fromkeys(seq))).reset_index()

    # Convert to list-of-lists: [uid, vid1, vid2, ...]
    uid_vid_list = grouped.apply(lambda r: [int(r['uid'])] + [int(v) for v in r['iid']], axis=1).tolist()
    print(f"\nBuilt uid_vid_list for {len(uid_vid_list)} uids. Showing first 5 users with first 20 interactions:")
    
    for row in uid_vid_list[:5]:
        print(row[:20])
        
    return uid_vid_list

big = build_uid_vid_list(train_df)
with open('./cleaned_data/big_matrix.txt', 'w', encoding='utf-8') as f:
    for row in big:
        f.write(' '.join(map(str, row)) + '\n')

raw_small = build_uid_vid_list(test_df)
with open('./cleaned_data/raw_small_matrix.txt', 'w', encoding='utf-8') as f:
    for row in raw_small:
        f.write(' '.join(map(str, row)) + '\n')

user_median = test_df.groupby('uid')['rating'].transform('median')
small = build_uid_vid_list(test_df[test_df['rating'] >= 4])
with open('./cleaned_data/small_matrix.txt', 'w', encoding='utf-8') as f:
    for row in small:
        f.write(' '.join(map(str, row)) + '\n')


Built uid_vid_list for 290 uids. Showing first 5 users with first 20 interactions:
[0, 72, 136, 150, 171, 188, 220, 227, 228, 234, 235, 236, 246, 247, 248, 250, 251, 252, 259, 266]
[1, 46, 70, 84, 124, 159, 174, 201, 215, 217, 220, 223, 225, 233, 236, 247, 249, 251, 252, 253]
[2, 138, 155, 162, 164, 165, 169, 176, 185, 206, 222, 228, 236, 240, 244, 248, 251, 252, 255, 256]
[3, 114, 153, 180, 186, 187, 201, 202, 203, 219, 222, 226, 227, 237, 249, 252, 263, 268, 270, 276]
[4, 99, 102, 110, 157, 169, 180, 185, 222, 226, 230, 237, 240, 242, 246, 247, 249, 250, 251, 252]

Built uid_vid_list for 290 uids. Showing first 5 users with first 20 interactions:
[0, 12, 17, 74, 78, 92, 104, 127, 128, 133, 145, 198, 199, 226, 265, 282, 283]
[1, 1, 29, 44, 104, 107, 122, 128, 147, 164, 186, 189, 190, 196, 219, 264, 275]
[2, 25, 43, 68, 70, 121, 134, 152, 173, 180, 188, 210, 214, 218, 220, 243, 270]
[3, 3, 65, 75, 81, 90, 92, 149, 173, 181, 185, 208, 224, 230, 280, 289, 293]
[4, 11, 13, 33, 42, 57, 70

In [4]:
'''
***************************************************
Split train
***************************************************
'''

data = []
with open('./cleaned_data/big_matrix.txt', 'r') as f:
    for line in f:
        row = [int(x) for x in line.strip().split()]
        data.append(row)
print(f"Number of users for training: {len(data)}")


import random
import math

def train_test_split_per_user(data, test_ratio=0.2, seed=42):
    random.seed(seed)
    train_data, test_data = [], []
    train_size = 0
    test_size = 0
    
    for u_items in data:
        items = u_items[1:]  # Exclude user ID
        
        n_items = len(items)
        n_test = math.ceil(n_items * test_ratio)
        test_items = random.sample(items, n_test)
        train_items = [x for x in items if x not in test_items]
        train_size += len(train_items)
        train_data.append(u_items[:1] + train_items)

        test_size += len(test_items)
        test_data.append(u_items[:1] + test_items)

    print(f"Train size: {train_size}; Test size: {test_size}.")
    print(f"Train users: {len(train_data)}; Test users: {len(test_data)}.")
    return train_data, test_data

train_data, test_data = train_test_split_per_user(data, test_ratio=0.3, seed=42)
with open('./cleaned_data/train.txt', 'w', encoding='utf-8') as f:
    for row in train_data:
        f.write(' '.join(map(str, row)) + '\n')

with open('./cleaned_data/val.txt', 'w', encoding='utf-8') as f:
    for row in test_data:
        f.write(' '.join(map(str, row)) + '\n')

Number of users for training: 290
Train size: 4497; Test size: 2097.
Train users: 290; Test users: 290.
